## 1. Confirm the accelerator

Choose `Runtime → Change runtime type` and select **GPU** before running anything else.

In [ ]:
import subprocess

try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception as exc:
    raise RuntimeError(
        'GPU not detected. In Colab, open Runtime -> Change runtime type, select GPU, save, then rerun this cell.'
    ) from exc

## 2. Configure the workspace path

In [ ]:
from pathlib import Path

WAN2GP_ROOT = Path('/content/wan2gp').resolve()
print(f'Wan2GP will be installed to: {WAN2GP_ROOT}')

## 3. Download or update Wan2GP

Clone the repository (pull the latest changes if it already exists). Set `BRANCH` to the branch that contains the `wan2gp_server/` folder.

In [ ]:
import subprocess

repo_url = 'https://github.com/hoangthvn2201/Wan2GP'
BRANCH = 'main'   # <-- change to the branch that contains wan2gp_server/

if WAN2GP_ROOT.exists():
    print('Repository already exists. Updating...')
    subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(WAN2GP_ROOT)], check=True)

subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'pull', 'origin', BRANCH], check=True)

if not (WAN2GP_ROOT / 'wan2gp_server' / '__main__.py').exists():
    raise RuntimeError(
        f"Branch '{BRANCH}' does not contain the wan2gp_server/ folder. "
        "Set BRANCH (above) to the branch where wan2gp_server was pushed and rerun this cell."
    )
print(f"OK: wan2gp_server/ found on branch '{BRANCH}'.")


## 4. Install system dependencies

In [ ]:
import os, subprocess

env = os.environ.copy()
env['DEBIAN_FRONTEND'] = 'noninteractive'

subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=True, env=env)
subprocess.run([
    'sudo', 'apt-get', 'install', '-y', '--no-install-recommends',
    'ffmpeg', 'libglib2.0-0', 'libgl1', 'libportaudio2'
], check=True, env=env)

## 5. Install Python dependencies (one-time)

A single install covers everything: Wan2GP's `requirements.txt` (torch ecosystem, diffusers, ...) plus the small `wan2gp_server/requirements.txt` extras (fastapi, uvicorn, python-multipart). This cell takes several minutes.

In [ ]:
import os, subprocess, sys

env = os.environ.copy()
env.setdefault('DEBIAN_FRONTEND', 'noninteractive')

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True, env=env)

subprocess.run([sys.executable, '-m', 'pip', 'install',
    '--force-reinstall', '--no-deps',
    'torch==2.8.0', 'torchvision==0.23.0', 'torchaudio==2.8.0',
    '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True, env=env)

subprocess.run([sys.executable, '-m', 'pip', 'install', 'xformers==0.0.32.post2',
    '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True, env=env)

# Wan2GP requirements + the server's small extras (kept inline so this cell
# works even if wan2gp_server/requirements.txt moves; see that file for the source of truth)
subprocess.run([sys.executable, '-m', 'pip', 'install',
    '-r', str(WAN2GP_ROOT / 'requirements.txt'),
    'fastapi', 'uvicorn[standard]', 'python-multipart'], check=True, env=env)

print('Dependencies installed.')

### 5b. Force a headless matplotlib backend

In [ ]:
target = WAN2GP_ROOT / 'preprocessing/matanyone/tools/interact_tools.py'
needle = "matplotlib.use('TkAgg')"
replacement = "matplotlib.use('Agg')"

if not target.exists():
    print(f'Skipping: {target} not found.')
else:
    text = target.read_text()
    if replacement in text:
        print('Agg backend already set; no change needed.')
    elif needle in text:
        target.write_text(text.replace(needle, replacement, 1))
        print('Replaced TkAgg with Agg in interact_tools.py.')
    else:
        print('Backend call not found; no change made.')

## 6. Configure the server

Pick the default model presets and WanGP low-VRAM flags. The full preset list and every environment variable are documented in `wan2gp_server/README.md`.

In [ ]:
import secrets

PORT = 8000

SERVER_ENV = {
    'WAN2GP_ROOT': str(WAN2GP_ROOT),
    'WAN2GP_SERVER_PORT': str(PORT),

    # WanGP startup flags - profile 5 = lowest VRAM (good for the free T4)
    'WAN2GP_CLI_ARGS': '--profile 5',

    # Default model presets (T4-friendly). Bigger GPUs:
    #   T2I: 'qwen-image' | 'flux-dev'
    #   T2V: 'wan21-fusionix' | 'wan22-t2v' | 'ltx2-distilled' (video + audio)
    #   I2V: 'wan21-fusionix-i2v' | 'wan22-i2v' | 'ltx2-distilled-i2v'
    'WAN2GP_SERVER_T2I_MODEL': 'z-image-turbo',
    'WAN2GP_SERVER_T2V_MODEL': 'wan21-t2v-1.3b',
    'WAN2GP_SERVER_I2V_MODEL': 'wan21-fun-inp-1.3b',

    # Protects /v1/* when you expose the server through the tunnel in step 11
    'WAN2GP_SERVER_API_KEY': secrets.token_urlsafe(16),
}

print('Server will run on port', PORT)
print('API key:', SERVER_ENV['WAN2GP_SERVER_API_KEY'])

## 7. Start the server (background)

Launches `python -m wan2gp_server` as a background process. The HTTP API is up within seconds — the WanGP runtime and model weights load lazily on the **first** generation (the first job therefore spends a while in `progress.phase = "loading_model"` downloading checkpoints).

Logs go to `/content/wan2gp_server.log`.

In [ ]:
import os, subprocess, sys, time
import requests

LOG_PATH = '/content/wan2gp_server.log'
BASE_URL = f'http://localhost:{PORT}'

env = os.environ.copy()
env.update(SERVER_ENV)

log_file = open(LOG_PATH, 'a')
server_proc = subprocess.Popen(
    [sys.executable, '-m', 'wan2gp_server'],
    cwd=str(WAN2GP_ROOT), env=env,
    stdout=log_file, stderr=subprocess.STDOUT,
)
print(f'Server PID: {server_proc.pid} | logs: {LOG_PATH}')

for _ in range(60):
    if server_proc.poll() is not None:
        print(open(LOG_PATH).read()[-3000:])
        raise RuntimeError('Server exited early - see the log above.')
    try:
        health = requests.get(f'{BASE_URL}/health', timeout=2).json()
        print('Server is up:', health)
        break
    except requests.ConnectionError:
        time.sleep(1)
else:
    raise RuntimeError('Server did not come up within 60s - check the log.')

## 8. Connect the client and list the models

In [ ]:
import sys

if str(WAN2GP_ROOT) not in sys.path:
    sys.path.insert(0, str(WAN2GP_ROOT))

from wan2gp_server.client import Wan2GPServerClient

client = Wan2GPServerClient(BASE_URL, api_key=SERVER_ENV['WAN2GP_SERVER_API_KEY'])
client.wait_until_ready()

for m in client.models():
    marker = ' (default)' if m['is_default'] else ''
    print(f"{m['task']}  {m['id']:<22}{marker}  - {m['description']}")

## 9. (Optional) Preload models

Warm up models **before** the first real request — each preload runs a tiny warmup generation that downloads the checkpoint and loads the weights into VRAM, so your first real job doesn't stall for minutes in `loading_model`.

Enter the preset ids you want to preload (see the list printed in step 8). Notes:

- `PRELOAD_MODELS = []` preloads the server's three **default** presets (t2i + t2v + i2v).
- `PRELOAD_MODELS = None` skips preloading entirely (models then load lazily on first use).
- WanGP keeps **one** model in VRAM — the **last** id in the list stays loaded; the others get their checkpoints cached on disk.

In [ ]:
PRELOAD_MODELS = []   # [] = server defaults | None = skip | e.g. ['qwen-image', 'wan21-fusionix']

if PRELOAD_MODELS is None:
    print('Skipping preload - models will load lazily on first generation.')
else:
    jobs = client.preload(PRELOAD_MODELS)   # blocks and prints progress per model
    for j in jobs:
        print(f"{j['model']:<22} -> {j['status']}")

## 10. Text → Image

Submits a job and polls until it finishes. The **first** call downloads the model checkpoint (a few GB) — watch `tail -f /content/wan2gp_server.log` in another cell if you are curious.

In [ ]:
OUT_DIR = Path('/content/outputs'); OUT_DIR.mkdir(exist_ok=True)

job = client.text_to_image(
    'A cozy reading nook by a rainy window, warm lamp light, watercolor style',
    width=768, height=768,
    # model='qwen-image',          # override the default preset
    # seed=42, steps=8,
)
print('Job:', job['id'], '| queued at position', job['queue_position'])

job = client.wait_for(job['id'])
image_path = client.download(job, OUT_DIR)
print('Saved:', image_path)

from IPython.display import Image as IPyImage, display
display(IPyImage(filename=str(image_path), width=384))

## 11. Text → Video

`duration_seconds` is converted to the model's frame grid automatically (e.g. 4 s × 16 fps → 65 frames). Switching from the image model to the video model makes WanGP unload/reload weights — expected.

In [ ]:
job = client.text_to_video(
    'A paper boat drifting down a rainy street gutter, cinematic, shallow depth of field',
    width=832, height=480,
    duration_seconds=4,
    # model='ltx2-distilled',      # bigger GPUs: video WITH audio
)
job = client.wait_for(job['id'])
video_path = client.download(job, OUT_DIR)
print('Saved:', video_path)

from IPython.display import Video
Video(str(video_path), embed=True, width=384)

## 12. Image → Video

Animates a start image. Here we reuse the image generated in step 10 — the client uploads any local file as base64; URLs (`image='https://...'`) and server-side paths work too. Output size defaults to the input image's size.

In [ ]:
job = client.image_to_video(
    'The camera slowly zooms in while rain streaks the window, gentle light flicker',
    image=image_path,              # local file | http(s) URL | server path | 'asset:<id>'
    duration_seconds=3,
)
job = client.wait_for(job['id'])
i2v_path = client.download(job, OUT_DIR)
print('Saved:', i2v_path)

from IPython.display import Video
Video(str(i2v_path), embed=True, width=384)

## 13. (Optional) Expose a public URL

Starts a free Cloudflare quick tunnel so agents / other machines can call the API. Requests to `/v1/*` must send the `X-API-Key` header printed in step 6. Keep the cell running while you use the tunnel; press **Stop** when done.

In [ ]:
import os, re, subprocess

if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(['wget', '-q', '-O', '/usr/local/bin/cloudflared',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}',
     '--protocol', 'http2', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

print('Waiting for the tunnel URL...')
public_url = None
for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        print(f'\n🌐 Wan2GP Server API: {public_url}')
        print(f'   Docs:              {public_url}/docs')
        print(f"   API key header:    X-API-Key: {SERVER_ENV['WAN2GP_SERVER_API_KEY']}")
        print()
        print('Example:')
        print(f"  curl -s {public_url}/v1/models -H 'X-API-Key: {SERVER_ENV['WAN2GP_SERVER_API_KEY']}'")
        break

try:
    for line in iter(tunnel_proc.stdout.readline, ''):
        if not line:
            break
except KeyboardInterrupt:
    tunnel_proc.terminate()
    print('Tunnel stopped.')

## Notes & troubleshooting

- **API docs**: `http://localhost:8000/docs` (interactive OpenAPI UI).
- **Job lifecycle**: `queued → running → succeeded | failed | cancelled`. Poll `GET /v1/jobs/{id}`; outputs appear under `files[*].url` on success, the failure reason under `error`.
- **First generation is slow**: WanGP downloads the checkpoint, then keeps it in VRAM — later jobs with the same model are much faster. Batch same-model requests together; switching presets unloads/reloads weights.
- **One job at a time**: the GPU runs a single generation; extra submissions queue FIFO (`queue_position` in the job JSON).
- **Out of VRAM on T4**: keep the default presets and `--profile 5`; reduce `width`/`height` (each preset also auto-caps the pixel area).
- **Server log**: `/content/wan2gp_server.log`. Stop the server with `server_proc.terminate()`.
- **Add your own model preset**: drop a JSON file into a folder and set `WAN2GP_SERVER_PRESETS_DIR` — `model_type` can be any file name from Wan2GP's `defaults/*.json`. See `wan2gp_server/README.md`.